# Chat with a timeseries table

[Tutorial 05](../05-timeseries-management/timeseries-management.ipynb) built everything a model
needs to work on a table of readings: a description of what is in it, a validator that compiles a
query before it runs, a read-only connection, and an edit that rehearses before it commits.

This notebook hands all four to one and asks it questions in English.

It is the table-side counterpart of [tutorial 04](../04-chat-with-graph/chat-with-graph.ipynb), and the
two are worth reading together. The pipeline is identical — route, write, validate, run, answer —
and everything that differs between them is something the difference between a graph and a table
forces.

The route:

1. a table to talk to, and the block that describes it
2. one question, end to end
3. what the validator catches, and why an empty result is the dangerous case
4. what `notes` is for — the same question asked with and without it
5. an edit: rehearsed, read, and only then committed
6. a conversation, where "and the meeting room?" means something
7. the terminal chat

**Every cell below that calls a model is billed to your key.** The committed run is a few tenths of
a cent on the library default; the `CostMeter` total at the end says exactly. Outputs will not
reproduce exactly.

In [1]:
import random
from pathlib import Path

import pandas as pd

import btwin
from btwin import LLM, SQL, CostMeter, Cycle, Observation, Tool

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

DB = OUTPUT / "rooms.db"
TABLE = "observations"

llm = LLM.Constructor()
meter = CostMeter()

print("BTwin", btwin.__version__)
print("model:", llm.model_name)

C:\Users\massa\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Couldn't import dot_parser, loading of dot files will not be possible.


BTwin 0.5.7
model: google/gemini-2.5-flash-lite


## 1. A table to talk to

Tutorial 05's week of readings, with a second room added: an open office that fills up every
working day, and a meeting room used in bursts. Eight sensors, 15-minute intervals, Monday 3 March
to Sunday 9 March 2025.

No model is involved in building it.

In [2]:
STEP_MINUTES = 15
DAYS = 7
START = pd.Timestamp("2025-03-03T00:00:00Z")   # a Monday

ROOMS = {
    # prefix: (seats, how it is used)
    "OFF": (22, "steady"),   # open office: in at 8, out at 18
    "MTG": (8, "bursty"),    # meeting room: booked for an hour at a time
}

rng = random.Random(20260901)
rows = []

for step in range(DAYS * 24 * 60 // STEP_MINUTES):
    stamp = START + pd.Timedelta(minutes=STEP_MINUTES * step)
    hour = stamp.hour + stamp.minute / 60
    weekend = stamp.dayofweek >= 5
    iso = stamp.strftime("%Y-%m-%dT%H:%M:%SZ")

    for prefix, (seats, pattern) in ROOMS.items():
        if weekend or hour < 7.5 or hour > 18.5:
            occupancy = 0
        elif pattern == "steady":
            ramp = min(1.0, (hour - 7.5) / 1.5, (18.5 - hour) / 1.5)
            occupancy = round(seats * ramp * rng.uniform(0.78, 1.05))
        else:
            # Booked on the hour, for an hour, about half the time
            occupancy = round(seats * rng.uniform(0.5, 1.0)) if rng.random() < 0.45 else 0

        co2 = 420 + occupancy * (24 if prefix == "OFF" else 62) * rng.uniform(0.9, 1.1)
        power = 4.2 + occupancy * 0.28 * rng.uniform(0.9, 1.1) + (2.6 if occupancy else 0)
        temperature = (21.0 + rng.uniform(-0.4, 0.4)) if occupancy else (18.4 + rng.uniform(-0.8, 0.8))

        rows += [
            [f"{prefix}-TEMP-01",  "Temperature",      "degC",   round(temperature, 2), iso],
            [f"{prefix}-CO2-01",   "CO2Concentration", "ppm",    round(co2, 1),         iso],
            [f"{prefix}-POWER-01", "ElectricalPower",  "kW",     round(power, 3),       iso],
            [f"{prefix}-OCC-01",   "Occupancy",        "people", float(occupancy),      iso],
        ]

observations = pd.DataFrame(rows, columns=list(Observation.Template().columns))
Observation.SQLiteByDF(observations, str(DB), TABLE, ifExists="replace")

index = Observation.SQLiteIndex(str(DB), TABLE)
print(f"{index['rows']} rows, {index['columns'][0]['distinct']} sensors")

5376 rows, 8 sensors


The description the model will be given. Two halves: what `Observation.SQLiteIndex` read off the
data, and the `notes` you supply — the part a flat table cannot say about itself.

In [3]:
NOTES = (
    "A sensor identifier reads ROOM-QUANTITY-INDEX. The room codes are OFF = the open office "
    "and MTG = the meeting room; the quantity codes are TEMP, CO2, POWER and OCC.\n"
    "To ask about one room, match the prefix with LIKE 'MTG-%' or substr(\"sosa:madeBySensor\", 1, 3).\n"
    "Occupancy is a count of people, not a percentage.\n"
    "ElectricalPower is an INSTANTANEOUS kW reading, not energy. Readings are 15 minutes "
    "apart, so energy in kWh over a period is SUM(value) * 0.25 - summing the kW column on "
    "its own gives a number in no unit at all."
)

schema = Observation.SQLiteSchemaSummary(str(DB), TABLE, index, notes=NOTES)
print(schema["text"][schema["text"].index("VALUES"):])

VALUES (the complete contents of the columns short enough to list -
        match these exactly, they are the only ones in the table)
  "sosa:madeBySensor": 'MTG-CO2-01', 'MTG-OCC-01', 'MTG-POWER-01', 'MTG-TEMP-01', 'OFF-CO2-01', 'OFF-OCC-01', 'OFF-POWER-01', 'OFF-TEMP-01'
  "sosa:ObservedProperty": 'CO2Concentration', 'ElectricalPower', 'Occupancy', 'Temperature'
  "unit": 'degC', 'kW', 'people', 'ppm'

NOTES
  A column name containing ':' or a space MUST be double-quoted, e.g.
    SELECT "sosa:madeBySensor" FROM "observations"
  "timestamp" is ISO 8601 TEXT: it sorts and compares as text, and is
  read with strftime - strftime('%Y', ts) for the year, '%Y-%m' for the month.
  SQLite has no DATE type and no EXTRACT, DATEPART, DATE_TRUNC or TO_CHAR.
  Aggregate with SUM, AVG, MIN, MAX, COUNT and group with GROUP BY. There is no
  other table to join: everything is in this one.
  A sensor identifier reads ROOM-QUANTITY-INDEX. The room codes are OFF = the open office and MTG = the meeting

## 2. One question, end to end

`Cycle.SQLiteQueryByPrompt` is five steps, three of which call a model:

1. `Observation.SQLiteIndex` and 2. `SQLiteSchemaSummary` describe the table — no model
3. `Tool.SQLiteWriteSQL` turns the question into a query
4. `SQL.Validate` checks it; `Tool.SQLiteRepairSQL` rewrites it when it fails
5. `Observation.SQLiteFetch` runs it read-only — no model — then `Tool.SQLiteAnswer` words the rows

The facts come from the data, not from the model: the answer is written from the rows the query
returned and nothing else.

In [4]:
result = Cycle.SQLiteQueryByPrompt(
    str(DB), TABLE,
    "What was the highest CO2 reading in the meeting room, and when?",
    llm=llm, schema=schema, meter=meter,
)

print(result["answer"])
print("\n--- the query it wrote ---")
print(result["sql"])
print("\n--- the rows it was shown ---")
for row in result["rows"][:5]:
    print(" ", row)
print("\ngrounded in:", result["source"])
print("cost:", CostMeter.Describe(result["usage"]))

The highest CO2 reading in the meeting room was 962.9 ppm at 2025-03-04 17:30:00.

--- the query it wrote ---
SELECT
  strftime('%Y-%m-%d %H:%M:%S', timestamp) AS time,
  value
FROM
  observations
WHERE
  "sosa:ObservedProperty" = 'CO2Concentration' AND "sosa:madeBySensor" LIKE 'MTG-%'
ORDER BY
  value DESC
LIMIT 1

--- the rows it was shown ---
  {'time': '2025-03-04 17:30:00', 'value': 962.9}

grounded in: []
cost: 1399+114 tokens, $0.000185


`source` came back empty, and that is informative rather than broken. It is the table's answer to
"where did this come from" — the counterpart of `RDF.SourceNodes` naming graph nodes in tutorial 04
— and it can only name sensors the query actually *selected*. This one asked for a timestamp and a
value, so the sensor column never reached the result and there is nothing to attribute. A query
that groups by `"sosa:madeBySensor"` fills it in; one that collapses the table to a single number
is grounded in the whole table, and says so by returning `[]`.

## 3. What the validator catches

The query above was accepted first time. The interesting cases are the ones that are not, and they
can be shown without spending a call — `SQL.Validate` is deterministic.

In [5]:
for candidate in [
    'SELECT AVG(co2) FROM observations',
    'SELECT value FROM observations WHERE room = \'meeting\'',
    'DROP TABLE observations',
]:
    _, error = SQL.Validate(candidate, str(DB), 100, schema["columns"])
    print(f"{candidate[:46]:<48} {error[:66]}")

SELECT AVG(co2) FROM observations                SQLite rejected the query: no such column: co2. Columns available:
SELECT value FROM observations WHERE room = 'm   SQLite rejected the query: no such column: room. Columns available
DROP TABLE observations                          DROP is not allowed; write a SELECT (a WITH ... SELECT is fine).


Every one of those is refused **by name**, because SQLite is asked to `EXPLAIN` the query — compile
it in full, resolving every column, without running a row. That reason string is what goes back to
`Tool.SQLiteRepairSQL`, and a repair that is told "no such column: co2" has something to work with.

The dangerous case is the one no validator can see: a query that names only real columns, compiles
cleanly, runs, and matches nothing — because the `WHERE` clause filtered on a literal that is not
in the table.

In [6]:
plausible = ('SELECT value FROM observations '
             'WHERE "sosa:ObservedProperty" = \'CO2\' LIMIT 10')

checked, error = SQL.Validate(plausible, str(DB), 100, schema["columns"])
print("validator says:", repr(error) or "accepted")
print("rows returned :", len(Observation.SQLiteFetch(str(DB), checked)))

validator says: ''
rows returned : 0


`'CO2'` is spelled `'CO2Concentration'` in this table. The query is valid, it runs, and it returns
nothing — which reads exactly like an honest "there is no CO2 data".

Two things guard against it. The **VALUES** section of the grounding block lists every literal the
string columns actually contain, so the writer has no reason to guess. And when a `SELECT` comes
back empty anyway, the cycle sends it once more to the repair agent with that fact as the reason,
keeping the rewrite only if it finds rows — `emptyRetries=0` turns that off.

## 4. What `notes` is for

The grounding block's generated half describes the table's *shape*. It cannot describe what the
readings **mean**, because that is nowhere in the data — and the gap that matters here is not the
room codes (a model can often guess `MTG` from eight sensor names) but the units.

`ElectricalPower` is an instantaneous kW reading taken every 15 minutes. Adding those up does not
give kWh; it gives a number in no unit at all, four times too large. Nothing in the table says so.
Ask for energy against a block with the notes stripped out:

In [7]:
bare = Observation.SQLiteSchemaSummary(str(DB), TABLE, index)   # no notes

blind = Cycle.SQLiteQueryByPrompt(
    str(DB), TABLE, "How many kWh did the meeting room use on Wednesday 5 March 2025?",
    llm=llm, schema=bare, meter=meter,
)
print("WITHOUT notes:", blind["answer"])
print(blind["sql"], "\n")

told = Cycle.SQLiteQueryByPrompt(
    str(DB), TABLE, "How many kWh did the meeting room use on Wednesday 5 March 2025?",
    llm=llm, schema=schema, meter=meter,
)
print("WITH notes:   ", told["answer"])
print(told["sql"])

WITHOUT notes: The meeting room used 457.36 kWh on Wednesday 5 March 2025.
SELECT
  SUM(value) AS totalKWh
FROM
  observations
WHERE
  "sosa:ObservedProperty" = 'ElectricalPower' AND "sosa:madeBySensor" = 'MTG-POWER-01' AND strftime('%Y-%m-%d', timestamp) = '2025-03-05'
LIMIT 100 



WITH notes:    The meeting room used 114.34 kWh on Wednesday 5 March 2025.
SELECT SUM(value) * 0.25 AS totalKWh FROM observations WHERE "sosa:madeBySensor" LIKE 'MTG-%' AND "sosa:ObservedProperty" = 'ElectricalPower' AND strftime('%Y-%m-%d', timestamp) = '2025-03-05'
LIMIT 100


Both runs see the same table. Only the second has been told that a kW reading every quarter of an
hour has to be multiplied by 0.25 to become kWh — and the two answers differ by exactly the factor
of four that follows from not knowing it.

Neither answer is flagged. A unit error does not fail a validator, does not return zero rows, and
does not read any differently from a right answer; it is simply four times too big. That is what
`notes` is for, and it is why the field exists at all: everything else in the grounding block can
be read off the data, and this cannot.

## 5. An edit

`Cycle.SQLiteEditByPrompt` is the same pipeline with the safety catch turned the other way: there
the model writes a `SELECT` and may not write, here it writes one `INSERT`, `UPDATE` or `DELETE`
and nothing else.

Say the meeting room's occupancy sensor was reading low all Wednesday.

In [8]:
edit = Cycle.SQLiteEditByPrompt(
    str(DB), TABLE,
    "The occupancy sensor in the meeting room was reading 20% low all day on Wednesday "
    "5 March 2025. Scale that day's non-zero readings up by a quarter, rounded to whole people.",
    llm=llm, schema=schema, meter=meter,
)

print(edit["sql"])
print(f"\n{edit['changes']} row(s) would change   committed={edit['committed']}")
for row in edit["removed"][:3]:
    print("  -", row)
for row in edit["added"][:3]:
    print("  +", row)

UPDATE "observations" SET "value" = round(value * 1.25) WHERE "sosa:madeBySensor" LIKE 'MTG-OCC-%' AND "sosa:ObservedProperty" = 'Occupancy' AND strftime('%Y-%m-%d', "timestamp") = '2025-03-05' AND "value" != 0

13 row(s) would change   committed=False
  - {'sosa:madeBySensor': 'MTG-OCC-01', 'sosa:ObservedProperty': 'Occupancy', 'unit': 'people', 'value': 4.0, 'timestamp': '2025-03-05T08:45:00Z'}
  - {'sosa:madeBySensor': 'MTG-OCC-01', 'sosa:ObservedProperty': 'Occupancy', 'unit': 'people', 'value': 5.0, 'timestamp': '2025-03-05T07:30:00Z'}
  - {'sosa:madeBySensor': 'MTG-OCC-01', 'sosa:ObservedProperty': 'Occupancy', 'unit': 'people', 'value': 5.0, 'timestamp': '2025-03-05T10:15:00Z'}
  + {'sosa:madeBySensor': 'MTG-OCC-01', 'sosa:ObservedProperty': 'Occupancy', 'unit': 'people', 'value': 10.0, 'timestamp': '2025-03-05T18:00:00Z'}
  + {'sosa:madeBySensor': 'MTG-OCC-01', 'sosa:ObservedProperty': 'Occupancy', 'unit': 'people', 'value': 5.0, 'timestamp': '2025-03-05T08:45:00Z'}
  + {'sosa:

Nothing has been written. With neither `inPlace` nor `savePath` the edit is a **dry run**: the
statement was validated, run inside a transaction, diffed, and rolled back. That is the mode to
read before trusting.

`SQL.ValidateUpdate` is what stood between that statement and the table. It refuses the three ways
a write can destroy data while answering the request exactly as put:

In [9]:
for candidate in [
    'DELETE FROM observations',
    "REPLACE INTO observations VALUES ('a', 'b', 'c', 1.0, 'd')",
    'UPDATE otherTable SET value = 0 WHERE 1',
]:
    _, error = SQL.ValidateUpdate(candidate, TABLE, str(DB), schema["columns"])
    print(f"{candidate[:44]:<46} {error[:70]}")

DELETE FROM observations                       A DELETE must carry a WHERE clause naming the rows to change. Without 
REPLACE INTO observations VALUES ('a', 'b',    REPLACE deletes the row it collides with. Write a plain INSERT to add 
UPDATE otherTable SET value = 0 WHERE 1        This update writes to 'otherTable'. It may only write to 'observations


To keep the edit, name somewhere for it to go. `savePath` copies the database and commits to the
copy, so the readings you started from are still there to compare against:

In [10]:
kept = Cycle.SQLiteEditByPrompt(
    str(DB), TABLE,
    "The occupancy sensor in the meeting room was reading 20% low all day on Wednesday "
    "5 March 2025. Scale that day's non-zero readings up by a quarter, rounded to whole people.",
    llm=llm, schema=schema, meter=meter,
    savePath=str(OUTPUT / "rooms_corrected.db"),
)

peak = ('SELECT MAX(value) AS peak FROM observations '
        'WHERE "sosa:madeBySensor" = \'MTG-OCC-01\' '
        '  AND strftime(\'%Y-%m-%d\', timestamp) = \'2025-03-05\'')

print(f"committed={kept['committed']} to {Path(kept['database']).name}")
print("original :", Observation.SQLiteFetch(str(DB), peak)[0]["peak"])
print("corrected:", Observation.SQLiteFetch(kept["database"], peak)[0]["peak"])

committed=True to rooms_corrected.db
original : 8.0
corrected: 10.0


## 6. A conversation

Both cycles above take one self-contained prompt and remember nothing. `Cycle.SQLiteChatTurn` puts
a conversation on top of them without changing that.

Every turn is routed first: `Tool.ChatRoute` reads the transcript and the new message and returns
an intent plus a **restatement that stands on its own**. That restatement is what reaches the query
cycle, so the pipeline underneath stays stateless and an answer is still written from retrieved
rows only. The memory is spent entirely on the rewrite.

In [11]:
history = []
schemaNow = schema

for message in [
    "What was the highest CO2 reading in the meeting room this week?",
    "and in the open office?",
    "which of the two rooms was busier on Wednesday?",
]:
    turn = Cycle.SQLiteChatTurn(
        str(DB), TABLE, message,
        history=history, llm=llm, schema=schemaNow, notes=NOTES, meter=meter,
    )
    history, schemaNow = turn["history"], turn["schema"]

    print(f"you> {message}")
    print(f"     understood as ({turn['intent']}): {turn['request']}")
    print(f"bot> {turn['answer']}")
    print(f"     {' '.join(turn['sql'].split())[:150]}\n")

you> What was the highest CO2 reading in the meeting room this week?
     understood as (question): What was the highest CO2 reading in the meeting room this week?
bot> The highest CO2 reading in the meeting room this week was 962.9 ppm.
     SELECT MAX(value) AS highestCO2Reading FROM observations WHERE "sosa:ObservedProperty" = 'CO2Concentration' AND "sosa:madeBySensor" LIKE 'MTG-%' AND s



you> and in the open office?
     understood as (question): What was the highest CO2 reading in the open office this week?
bot> The highest CO2 reading in the open office this week was 1005.7.
     SELECT MAX(value) AS highestCO2Reading FROM observations WHERE "sosa:ObservedProperty" = 'CO2Concentration' AND "sosa:madeBySensor" LIKE 'OFF-%' AND s



you> which of the two rooms was busier on Wednesday?
     understood as (question): Which of the two rooms, the meeting room or the open office, had higher CO2 readings on Wednesday?
bot> The open office had higher CO2 readings on Wednesday, with an average of 613.0. The meeting room had an average CO2 reading of 466.44166666666666.
     SELECT CASE substr("sosa:madeBySensor", 1, 3) WHEN 'MTG' THEN 'meeting room' WHEN 'OFF' THEN 'open office' END AS room, AVG(value) AS average_co2 FROM



The second message is the whole point. *"and in the open office?"* is not a question anyone could
answer on its own — the router resolves it against the transcript into something that is, and the
query cycle never sees the ellipsis.

The third message shows the same mechanism being a little too helpful. "Busier" is not defined
anywhere, and the router settled it from context: two turns about CO2, so *busier* was restated as
*higher CO2 readings*. Defensible, and not what everyone means by busy — which is exactly why the
restatement is printed on its own line and why every turn shows the SQL underneath. The place a
misunderstanding is visible is the query, not the sentence.

This is what the transcript looks like from the router's side:

In [12]:
print(Tool.ChatTranscript(history))

[1] user: What was the highest CO2 reading in the meeting room this week?
     assistant: The highest CO2 reading in the meeting room this week was 962.9 ppm.
[2] user: and in the open office?
     understood as (question): What was the highest CO2 reading in the open office this week?
     assistant: The highest CO2 reading in the open office this week was 1005.7.
[3] user: which of the two rooms was busier on Wednesday?
     understood as (question): Which of the two rooms, the meeting room or the open office, had higher CO2 readings on Wednesday?
     assistant: The open office had higher CO2 readings on Wednesday, with an average of 613.0. The meeting room had an average CO2 reading of 466.44166666666666.


## 7. The terminal chat

`Cycle.SQLiteChat` is the loop around all of that, and the only thing in the module that reads a
keyboard. It blocks on input, so it is described here rather than run:

```python
session = Cycle.SQLiteChat(
    "output/rooms.db", "observations",
    notes=NOTES,
    savePath="output/rooms_session.db",   # edits go to a copy; the original is never written
)
print(session["edits"], session["database"])
```

Where a confirmed edit lands is decided once, before the first turn, and said out loud:

| | |
|---|---|
| `savePath` | the database is copied there and the whole conversation talks to the copy |
| `inPlace=True` | the conversation edits the original — there is no undo |
| neither | edits are rehearsed and shown, but cannot be kept, and the session says so |

Every turn prints the SQL it ran and what it cost, because the reading that catches a wrong answer
is the query rather than the sentence — section 3 is why. The commands are `/sql`, `/rows`,
`/history`, `/schema`, `/cost`, `/silent`, `/verbose` and `/exit`; Escape leaves too, and at an edit
confirmation it means no.

There is no `/save`, and that is the one place a database and a graph genuinely part company: a
committed edit is already on disk, because a database *is* the file rather than something
serialised into one.

`myproject/chat_with_sqlite.py` in this repository is a worked use case around it, including a
session log that writes out every model call and every deterministic step in between.

## What this run cost

In [13]:
total = meter.Total()
print(f"{total['calls']} call(s)")
print(f"  prompt     {total['promptTokens']:>7} tokens")
print(f"  completion {total['completionTokens']:>7} tokens")
print(f"  cost       {CostMeter.Format(total['cost'])}"
      + ("  (estimated)" if total["estimated"] else ""))

17 call(s)
  prompt       11812 tokens
  completion     917 tokens
  cost       $0.001548


## What you have

`output/rooms.db` — a week of readings from two rooms — and `output/rooms_corrected.db`, the same
week with Wednesday's meeting-room occupancy scaled up. Two files that can be diffed, which is the
only reason to prefer a copy over an in-place edit.

And a pipeline where every step a model took is inspectable after the fact: the SQL it wrote, the
rows it was shown, the sensors those rows came from, and what each call cost.

| | Graph | Table |
|---|---|---|
| describe | `RDF.SchemaSummary` + `RDF.Chains` | `Observation.SQLiteSchemaSummary` |
| ask | `Cycle.RDFQueryByPrompt` | `Cycle.SQLiteQueryByPrompt` |
| check | `SPARQL.Validate` | `SQL.Validate` (compiles, via `EXPLAIN`) |
| change | `Cycle.RDFEditByPrompt` | `Cycle.SQLiteEditByPrompt` (rehearses, then commits) |
| converse | `Cycle.RDFChat` | `Cycle.SQLiteChat` |

[Tutorial 04](../04-chat-with-graph/chat-with-graph.ipynb) is the left-hand column.